In [2]:
import torch

from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import LlamaTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain import hub

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=100)

embedding = HuggingFaceEmbeddings()

/home/nhatthuong/.miniconda3/envs/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
def process_file(path):
    # if file.type == "text/plain":
    #     Loader = TextLoader
    # elif file.type == "application/pdf":
    #     Loader = PyPDFLoader
    Loader = PyPDFLoader
    loader = Loader(path)
    documents = loader.load()
    docs = text_splitter.split_documents(documents)
    for i, doc in enumerate(docs):
        doc.metadata["source"] = f"source_{i}"
    return docs

In [15]:
process_file('/home/nhatthuong/Documents/Thesis/code/backend/fastapi_ai/rag_acne/infoacne.pdf')

[Document(metadata={'source': 'source_0', 'page': 0}, page_content='1.\nAcne\nVulgaris\nDescription:\nAcne\nvulgaris\nis\nthe\nmost\ncommon\nform\nof\nacne,\ncharacterized\nby\nthe\npresence\nof\nblackheads,\nwhiteheads,\npapules,\npustules,\nnodules,\nand\ncysts.\nIt\ntypically\noccurs\non\nthe\nface,\nback,\nand\nshoulders.\nTreatment:\n●\nTopical\nTreatments:\nBenzoyl\nperoxide,\nsalicylic\nacid,\nretinoids,\nand\nantibiotics.\n●\nOral\nMedications:\nAntibiotics,\nhormonal\ntreatments\n(like\nbirth\ncontrol\npills),\nand\nisotretinoin\nfor\nsevere\ncases.\n●\nLifestyle\nChanges:\nRegular\ncleansing,\navoiding\nheavy\nmakeup,\nand\na\nbalanced\ndiet.\n2.\nBirthmarks\nDescription:\nBirthmarks\nare\nbenign\nirregularities\non\nthe\nskin\nthat\nare\npresent\nat\nbirth\nor\nappear\nshortly\nafterward.\nThey\nare\nnot\na\nform\nof\nacne\nbut\nare\nincluded\nhere\nfor\nclarity .\nTreatment:\n●\nObservation:\nMost\nbirthmarks\nare\nharmless\nand\ndo\nnot\nrequire\ntreatment.\n●\nLaser\nTher

In [5]:
def get_huggingface_llm(model_name: str = "lmsys/vicuna-7b-v1.5", max_new_token: int = 512):
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=nf4_config,
        low_cpu_mem_usage=True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_token,
        pad_token_id=tokenizer.eos_token_id,
        device_map="auto"
    )

    llm = HuggingFacePipeline(
        pipeline=model_pipeline,
    )
    return llm

LLM = get_huggingface_llm()

Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.76s/it]
/home/nhatthuong/.miniconda3/envs/myenv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/nhatthuong/.miniconda3/envs/myenv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect 

In [16]:
welcome_message = """Welcome to the PDF QA! To get started:
1. Upload a PDF or text file
2. Ask a question about the file
"""

: 

In [ ]:
def on_chat_start():
    # Replace Chainlit's file request with a standard file input method
    print("Please provide the path to the file you want to process (text/plain or application/pdf):")
    file_path = input("File path: ")
    
    # Open and read the file
    try:
        with open(file_path, 'rb') as f:
            file_content = f.read()
        file_name = file_path.split('/')[-1]
    except FileNotFoundError:
        print("File not found. Please check the path and try again.")
        return

    print(f"Processing `{file_name}`...")

    # Replace `get_vector_db` with your own logic to process the file content
    vector_db = get_vector_db(file_content)

    # Initialize message history and memory
    message_history = ChatMessageHistory()
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        output_key="answer",
        chat_memory=message_history,
        return_messages=True,
    )

    # Create a retriever from the vector database
    retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={'k': 3})

    # Set up the conversational retrieval chain
    chain = ConversationalRetrievalChain.from_llm(
        llm=LLM,
        chain_type="stuff",
        retriever=retriever,
        memory=memory,
        return_source_documents=True
    )

    print(f"`{file_name}` processed. You can now ask questions!")

    # Store the chain in a global or session-like context if needed
    global_session["chain"] = chain  # Replace with your session management logic

# Example of a simple global session dictionary for demonstration
global_session = {}


In [ ]:
def on_message(message: str):
    # Retrieve the chain from the user session (replace with your own session management)
    chain = cl.user_session.get("chain")
    
    res = chain.invoke(message)  # Adjust this line based on how `invoke` is implemented

    # Extract the answer and source documents
    answer = res["answer"]
    source_documents = res["source_documents"]
    text_elements = []

    # Process source documents if available
    if source_documents:
        for source_idx, source_doc in enumerate(source_documents):
            source_name = f"source_{source_idx}"
            text_elements.append({
                "content": source_doc.page_content,
                "name": source_name
            })
        source_names = [text_el["name"] for text_el in text_elements]

        if source_names:
            answer += f"\nSources: {', '.join(source_names)}"
        else:
            answer += "\nNo sources found"

    # Send the message (replace with your own messaging logic)
    # This is a placeholder for sending the message, adjust accordingly
    print(f"Answer: {answer}")
    for element in text_elements:
        print(f"Source Document: {element['name']} - Content: {element['content']}")
